# CCR-Tabular — Results Visualization

Generates all paper figures from `outputs/metrics/results.csv`.

**Run this after all experiments complete.**

Outputs saved to `outputs/plots/`.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.utils.config import OUTPUTS_METRICS, OUTPUTS_PLOTS, DATASETS, MODEL_NAMES
from src.utils.statistics import run_all_wilcoxon_tests

# ── Style ──────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})

PLOTS = OUTPUTS_PLOTS
PLOTS.mkdir(parents=True, exist_ok=True)

# ── Color palette ──────────────────────────────────────────────────────────
MODEL_COLORS = {
    'mlp_standard':    '#95a5a6',
    'mlp_focal':       '#3498db',
    'mlp_weighted_ce': '#2ecc71',
    'mlp_smote':       '#e67e22',
    'xgboost_default': '#9b59b6',
    'xgboost_weighted':'#8e44ad',
    'lightgbm_default':'#1abc9c',
    'mlp_ccr':         '#e74c3c',
}

MODEL_LABELS = {
    'mlp_standard':    'MLP (CE)',
    'mlp_focal':       'MLP (Focal)',
    'mlp_weighted_ce': 'MLP (WCE)',
    'mlp_smote':       'MLP (SMOTE)',
    'xgboost_default': 'XGBoost',
    'xgboost_weighted':'XGBoost (W)',
    'lightgbm_default':'LightGBM',
    'mlp_ccr':         'CCR (Ours)',
}

print('Setup complete.')

In [ ]:
# ── Load results ────────────────────────────────────────────────────────────
results_path = OUTPUTS_METRICS / 'results.csv'
assert results_path.exists(), f'results.csv not found at {results_path}. Run experiments first.'

df = pd.read_csv(results_path)
print(f'Loaded {len(df)} rows from results.csv')
print(f'Datasets: {sorted(df["dataset"].unique())}')
print(f'Models:   {sorted(df["model"].unique())}')
print(f'Noise:    {df[["noise_type","noise_rate"]].drop_duplicates().to_string(index=False)}')
df.head(3)

In [ ]:
# ── Helper: aggregate mean ± std per model per dataset ─────────────────────
def aggregate(df_in, noise_type='none', noise_rate=0.0, metric='macro_f1'):
    mask = (df_in['noise_type'] == noise_type) & (df_in['noise_rate'].round(3) == round(noise_rate, 3))
    sub = df_in[mask].groupby(['dataset', 'model'])[metric].agg(['mean', 'std']).reset_index()
    sub.columns = ['dataset', 'model', 'mean', 'std']
    return sub

print('Helper defined.')

## Figure 1 — Macro F1 on Clean Data (All Datasets × All Models)

In [ ]:
agg = aggregate(df, noise_type='none', noise_rate=0.0, metric='macro_f1')
datasets = sorted(agg['dataset'].unique())
models = [m for m in MODEL_NAMES if m in agg['model'].unique()]

fig, axes = plt.subplots(1, len(datasets), figsize=(4 * len(datasets), 5), sharey=False)
if len(datasets) == 1:
    axes = [axes]

for ax, dataset in zip(axes, datasets):
    sub = agg[agg['dataset'] == dataset].set_index('model')
    means = [sub.loc[m, 'mean'] if m in sub.index else 0 for m in models]
    stds  = [sub.loc[m, 'std']  if m in sub.index else 0 for m in models]
    colors = [MODEL_COLORS.get(m, '#aaa') for m in models]
    labels = [MODEL_LABELS.get(m, m) for m in models]

    bars = ax.bar(range(len(models)), means, yerr=stds, capsize=3,
                  color=colors, edgecolor='white', linewidth=0.5)
    # Highlight CCR bar
    if 'mlp_ccr' in models:
        idx = models.index('mlp_ccr')
        bars[idx].set_edgecolor('black')
        bars[idx].set_linewidth(2)

    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_title(dataset, fontweight='bold')
    ax.set_ylabel('Macro F1' if ax == axes[0] else '')
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

fig.suptitle('Macro F1 — Clean Labels', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(PLOTS / 'fig_01_macro_f1_clean.png')
fig.savefig(PLOTS / 'fig_01_macro_f1_clean.pdf')
plt.show()
print('Saved fig_01_macro_f1_clean')

## Figure 2 — Minority Recall on Clean Data

In [ ]:
agg_recall = aggregate(df, noise_type='none', noise_rate=0.0, metric='minority_recall')

fig, axes = plt.subplots(1, len(datasets), figsize=(4 * len(datasets), 5), sharey=False)
if len(datasets) == 1:
    axes = [axes]

for ax, dataset in zip(axes, datasets):
    sub = agg_recall[agg_recall['dataset'] == dataset].set_index('model')
    means = [sub.loc[m, 'mean'] if m in sub.index else 0 for m in models]
    stds  = [sub.loc[m, 'std']  if m in sub.index else 0 for m in models]
    colors = [MODEL_COLORS.get(m, '#aaa') for m in models]

    bars = ax.bar(range(len(models)), means, yerr=stds, capsize=3,
                  color=colors, edgecolor='white', linewidth=0.5)
    if 'mlp_ccr' in models:
        idx = models.index('mlp_ccr')
        bars[idx].set_edgecolor('black')
        bars[idx].set_linewidth(2)

    ax.set_xticks(range(len(models)))
    ax.set_xticklabels([MODEL_LABELS.get(m, m) for m in models], rotation=45, ha='right', fontsize=8)
    ax.set_title(dataset, fontweight='bold')
    ax.set_ylabel('Minority Recall' if ax == axes[0] else '')
    ax.set_ylim(0, 1.05)

fig.suptitle('Minority-Class Recall — Clean Labels', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(PLOTS / 'fig_02_minority_recall_clean.png')
fig.savefig(PLOTS / 'fig_02_minority_recall_clean.pdf')
plt.show()
print('Saved fig_02_minority_recall_clean')

## Figure 3 — Noise Degradation Curves (CCR vs Top Baselines)

In [ ]:
noise_rates = sorted(df['noise_rate'].unique())
focus_models = ['mlp_ccr', 'mlp_focal', 'mlp_weighted_ce', 'xgboost_weighted']
focus_models = [m for m in focus_models if m in df['model'].unique()]

for noise_type in ['asym', 'feat']:
    noise_df = df[df['noise_type'] == noise_type]
    if len(noise_df) == 0:
        print(f'No data for noise_type={noise_type}, skipping.')
        continue

    available_rates = sorted(noise_df['noise_rate'].unique())
    if len(available_rates) == 0:
        continue

    fig, axes = plt.subplots(1, len(datasets), figsize=(4.5 * len(datasets), 4.5), sharey=False)
    if len(datasets) == 1:
        axes = [axes]

    for ax, dataset in zip(axes, datasets):
        for model in focus_models:
            means, stds, rates_used = [], [], []
            for rate in available_rates:
                sub = noise_df[
                    (noise_df['dataset'] == dataset) &
                    (noise_df['model'] == model) &
                    (noise_df['noise_rate'].round(3) == round(rate, 3))
                ]['macro_f1']
                if len(sub) > 0:
                    means.append(sub.mean())
                    stds.append(sub.std())
                    rates_used.append(rate)

            if means:
                lw = 2.5 if model == 'mlp_ccr' else 1.5
                ls = '-' if model == 'mlp_ccr' else '--'
                ax.plot(rates_used, means,
                        label=MODEL_LABELS.get(model, model),
                        color=MODEL_COLORS.get(model, '#aaa'),
                        linewidth=lw, linestyle=ls, marker='o', markersize=5)
                ax.fill_between(rates_used,
                                [m - s for m, s in zip(means, stds)],
                                [m + s for m, s in zip(means, stds)],
                                alpha=0.12,
                                color=MODEL_COLORS.get(model, '#aaa'))

        ax.set_title(dataset, fontweight='bold')
        ax.set_xlabel('Noise Rate')
        ax.set_ylabel('Macro F1' if ax == axes[0] else '')
        ax.set_xlim(-0.01, max(available_rates) + 0.02)
        ax.set_ylim(0, 1.0)
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        if ax == axes[-1]:
            ax.legend(fontsize=7, loc='lower left')

    noise_label = 'Asymmetric' if noise_type == 'asym' else 'Feature-Correlated'
    fig.suptitle(f'Macro F1 Degradation — {noise_label} Noise', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(PLOTS / f'fig_03_degradation_{noise_type}.png')
    fig.savefig(PLOTS / f'fig_03_degradation_{noise_type}.pdf')
    plt.show()
    print(f'Saved fig_03_degradation_{noise_type}')

## Figure 4 — Training Time: CCR vs Baselines

In [ ]:
if 'train_time_s' in df.columns:
    time_df = df[df['noise_type'] == 'none'].groupby(['dataset', 'model'])['train_time_s'].mean().reset_index()

    fig, axes = plt.subplots(1, len(datasets), figsize=(4 * len(datasets), 4.5), sharey=False)
    if len(datasets) == 1:
        axes = [axes]

    for ax, dataset in zip(axes, datasets):
        sub = time_df[time_df['dataset'] == dataset].set_index('model')
        times = [sub.loc[m, 'train_time_s'] if m in sub.index else 0 for m in models]
        colors = [MODEL_COLORS.get(m, '#aaa') for m in models]

        bars = ax.bar(range(len(models)), times, color=colors, edgecolor='white')
        if 'mlp_ccr' in models:
            idx = models.index('mlp_ccr')
            bars[idx].set_edgecolor('black')
            bars[idx].set_linewidth(2)

        ax.set_xticks(range(len(models)))
        ax.set_xticklabels([MODEL_LABELS.get(m, m) for m in models], rotation=45, ha='right', fontsize=8)
        ax.set_title(dataset, fontweight='bold')
        ax.set_ylabel('Training Time (s)' if ax == axes[0] else '')

    fig.suptitle('Mean Training Time per Run (seconds)', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(PLOTS / 'fig_04_training_time.png')
    fig.savefig(PLOTS / 'fig_04_training_time.pdf')
    plt.show()
    print('Saved fig_04_training_time')
else:
    print('train_time_s column not found in results.csv. Re-run experiments to collect timing data.')

## Figure 5 — Heatmap: CCR Improvement over Best Baseline

In [ ]:
metrics_to_show = ['macro_f1', 'minority_recall', 'auc_roc']
noise_conditions = [('none', 0.0), ('asym', 0.2), ('feat', 0.2)]

for noise_type, noise_rate in noise_conditions:
    mask = (df['noise_type'] == noise_type) & (df['noise_rate'].round(3) == round(noise_rate, 3))
    sub = df[mask]
    if len(sub) == 0:
        continue

    rows_data = []
    for dataset in sorted(sub['dataset'].unique()):
        for metric in metrics_to_show:
            ccr_mean = sub[(sub['dataset'] == dataset) & (sub['model'] == 'mlp_ccr')][metric].mean()
            best_baseline = sub[
                (sub['dataset'] == dataset) & (sub['model'] != 'mlp_ccr')
            ].groupby('model')[metric].mean().max()
            delta = ccr_mean - best_baseline if not np.isnan(ccr_mean) and not np.isnan(best_baseline) else np.nan
            rows_data.append({'dataset': dataset, 'metric': metric, 'delta': delta})

    heat_df = pd.DataFrame(rows_data).pivot(index='dataset', columns='metric', values='delta')

    fig, ax = plt.subplots(figsize=(6, max(3, len(heat_df) * 0.7)))
    sns.heatmap(
        heat_df[metrics_to_show], annot=True, fmt='.3f', center=0,
        cmap='RdYlGn', linewidths=0.5, ax=ax,
        cbar_kws={'label': 'CCR − Best Baseline'}
    )
    noise_label = f'{noise_type}@{noise_rate:.0%}' if noise_type != 'none' else 'clean'
    ax.set_title(f'CCR Improvement over Best Baseline ({noise_label})', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout()
    fname = f'fig_05_improvement_heatmap_{noise_type}_{int(noise_rate*100):02d}'
    fig.savefig(PLOTS / f'{fname}.png')
    fig.savefig(PLOTS / f'{fname}.pdf')
    plt.show()
    print(f'Saved {fname}')

## Statistical Significance (Wilcoxon Tests)

In [ ]:
try:
    all_wilcoxon = run_all_wilcoxon_tests()
    print(f'\nWilcoxon tests computed for {len(all_wilcoxon)} noise conditions.')
    for key, wdf in all_wilcoxon.items():
        n_sig = wdf['significant'].sum()
        n_better = wdf['better'].sum()
        print(f'  {key}: {n_sig}/{len(wdf)} significant, {n_better}/{len(wdf)} CCR better (p<0.05)')
except Exception as e:
    print(f'Wilcoxon tests skipped: {e}')

## Summary Table (Paper-Ready)

In [ ]:
# Clean condition summary table
clean = df[df['noise_type'] == 'none']
if len(clean) > 0:
    summary = clean.groupby(['dataset', 'model'])[['macro_f1', 'minority_recall', 'auc_roc']].agg(
        lambda x: f'{x.mean():.3f} ± {x.std():.3f}'
    ).reset_index()
    summary['model'] = summary['model'].map(MODEL_LABELS).fillna(summary['model'])
    print('Clean condition results (Mean ± Std):')
    display(summary.pivot(index='model', columns='dataset', values='macro_f1'))

    # Save to CSV
    summary.to_csv(OUTPUTS_METRICS / 'paper_table_clean.csv', index=False)
    print('Saved paper_table_clean.csv')